In [23]:
pip install tensorflow

  Using cached tensorflow-2.20.0-cp313-cp313-win_amd64.whl.metadata (4.6 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached setuptools-82.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached wrapt-2.1.1-cp313-cp313-win_amd64.whl.metadata (7.6 kB)
  Using cached grpcio-1.78.0-cp313-cp313-win_amd64.whl.metadata (3.9 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached 

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mahmoudreda55/satellite-image-classification")

print("Path to dataset files:", path)

c:\Users\Avantee Sarve\OneDrive\Desktop\college\Green AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\Avantee Sarve\.cache\kagglehub\datasets\mahmoudreda55\satellite-image-classification\versions\1


In [5]:
import os
import pandas as pd

labels = {}

for root, _, files in os.walk(path):
    if any(f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")) for f in files):
        labels[root] = os.path.basename(root)

data = []

for folder in labels:
    if not os.path.exists(folder):
        print(f"Warning: The folder {folder} does not exist.")
        continue

    for image_name in os.listdir(folder):
        image_path = os.path.join(folder, image_name)

        if os.path.isfile(image_path):
            data.append({
                "image_path": image_path,
                "label": labels[folder]
            })

data = pd.DataFrame(data)

print(data)

                                             image_path   label
0     C:\Users\Avantee Sarve\.cache\kagglehub\datase...  cloudy
1     C:\Users\Avantee Sarve\.cache\kagglehub\datase...  cloudy
2     C:\Users\Avantee Sarve\.cache\kagglehub\datase...  cloudy
3     C:\Users\Avantee Sarve\.cache\kagglehub\datase...  cloudy
4     C:\Users\Avantee Sarve\.cache\kagglehub\datase...  cloudy
...                                                 ...     ...
5626  C:\Users\Avantee Sarve\.cache\kagglehub\datase...   water
5627  C:\Users\Avantee Sarve\.cache\kagglehub\datase...   water
5628  C:\Users\Avantee Sarve\.cache\kagglehub\datase...   water
5629  C:\Users\Avantee Sarve\.cache\kagglehub\datase...   water
5630  C:\Users\Avantee Sarve\.cache\kagglehub\datase...   water

[5631 rows x 2 columns]


In [6]:
data.to_csv("satellite_images_labels.csv", index=False)

In [7]:
image_df = pd.read_csv("satellite_images_labels.csv")
image_df.head()

,image_path,label
0,C:\Users\Avantee Sarve\.cache\kagglehub\datase...,cloudy
1,C:\Users\Avantee Sarve\.cache\kagglehub\datase...,cloudy
2,C:\Users\Avantee Sarve\.cache\kagglehub\datase...,cloudy
3,C:\Users\Avantee Sarve\.cache\kagglehub\datase...,cloudy
4,C:\Users\Avantee Sarve\.cache\kagglehub\datase...,cloudy


In [8]:
from sklearn.model_selection import train_test_split
x_train, x_test = train_test_split(data, test_size=0.2)

In [9]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
train_datagen = ImageDataGenerator(
  rescale=1./255,
  shear_range=0.2,
  zoom_range=0.2,
  horizontal_flip=True,
  rotation_range=45,
  vertical_flip=True,
  fill_mode='nearest'
)

test_datagen = ImageDataGenerator(rescale=1./255)

In [13]:
train_generator = train_datagen.flow_from_dataframe(
    x_train,
    x_col='image_path',
    y_col='label',
    target_size=(255, 255),
    batch_size=32,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_dataframe(
    x_test,
    x_col='image_path',
    y_col='label',
    target_size=(255, 255),
    batch_size=32,
    class_mode='categorical'
)

Found 4504 validated image filenames belonging to 4 classes.
Found 1127 validated image filenames belonging to 4 classes.


In [23]:
# CNN architecture
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Dropout
sat_cnn_model = Sequential()
sat_cnn_model.add(Conv2D(100, (3,3), activation='relu', input_shape=(255,255,3)))
sat_cnn_model.add(MaxPooling2D((2,2)))
sat_cnn_model.add(Conv2D(50, (3,3), activation='relu'))
sat_cnn_model.add(MaxPooling2D((2,2)))
#sat_cnn_model.add(Conv2D(128, (3,3), activation='relu'))
sat_cnn_model.add(Flatten())
sat_cnn_model.add(Dense(50, activation='relu'))
sat_cnn_model.add(Dropout(0.5))
sat_cnn_model.add(Dense(4, activation='softmax'))
sat_cnn_model.summary()

c:\Users\Avantee Sarve\OneDrive\Desktop\college\Green AI\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_8 (Conv2D)               │ (None, 253, 253, 100)  │         2,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 126, 126, 100)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 124, 124, 50)   │        45,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 62, 62, 50)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 192200)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 50)             │     9,610,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 4)              │           204 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,658,104 (36.84 MB)

 Trainable params: 9,658,104 (36.84 MB)

 Non-trainable params: 0 (0.00 B)

In [24]:
sat_cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=10)
history = sat_cnn_model.fit(train_generator, epochs=1, validation_data=test_generator, callbacks=[early_stop])
sat_cnn_model.evaluate(test_generator)

141/141 ━━━━━━━━━━━━━━━━━━━━ 302s 2s/step - accuracy: 0.6530 - loss: 0.7388 - val_accuracy: 0.8217 - val_loss: 0.4380
36/36 ━━━━━━━━━━━━━━━━━━━━ 12s 326ms/step - accuracy: 0.8217 - loss: 0.4380


[0.4380105435848236, 0.8216503858566284]